In [3]:
!pip install -qU pandas numpy torch tqdm unsloth transformers peft

  error: subprocess-exited-with-error
  
  × Building wheel for xformers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [335 lines of output]
      /private/var/folders/gp/4zdmthz13v75n8wj6ldr_rmw0000gn/T/pip-build-env-f019co93/overlay/lib/python3.10/site-packages/torch/_subclasses/functional_tensor.py:279: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:81.)
        cpu = _conversion_method_template(device=torch.device("cpu"))
      /private/var/folders/gp/4zdmthz13v75n8wj6ldr_rmw0000gn/T/pip-build-env-f019co93/overlay/lib/python3.10/site-packages/setuptools/dist.py:759: SetuptoolsDeprecationWarning: License classifiers are deprecated.
      !!
      
              ********************************************************************************
              Please consider removing the following classifiers in favor of a SPDX license expression:
  

# Model Evaluation Notebook

This notebook is designed to evaluate and compare the performance of two fine-tuned models:
1.  **SFT-Only Model**: The model after Supervised Fine-Tuning.
2.  **GRPO Model**: The final model after training with GRPO.

We will evaluate them on a held-out test set using a comprehensive suite of metrics:
- **Classification Metrics (for `status`)**: Accuracy, Precision, Recall, F1-Score
- **Regression Metrics (for `score`)**: Mean Absolute Error (MAE), Root Mean Squared Error (RMSE)
- **Text Generation Metrics (for `explanation`)**: ROUGE-L

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import json
import re
import torch
from tqdm.auto import tqdm

# Unsloth, PEFT, and Transformers for model loading
from unsloth import FastLanguageModel
from transformers import AutoTokenizer
from peft import PeftModel

# Scikit-learn and ROUGE for metrics
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from rouge_score import rouge_scorer

## 2. Configuration
Set the paths for the models and the evaluation dataset.

In [ ]:
BASE_MODEL_NAME = "unsloth/Qwen2-0.5B-Instruct-bnb-4bit"
SFT_ADAPTER_PATH = "/models/sft_qwen_resume_eval_model"  
GRPO_ADAPTER_PATH = "/models/grpo_resume_model"      
FULL_DATA_PATH = "/content/o6ai-agent-hr/synthetic_resume_eval_data_1000.json" 

## 3. Data Loading and Preparation
We will load the full dataset and then create the `sft_dataset_eval` variable, which contains the last 100 samples that were used as the held-out test set.

In [ ]:
# Load the full dataset from the original file
full_df = pd.read_json(FULL_DATA_PATH)

# Create the evaluation dataset from the last 100 samples
sft_dataset_eval = full_df[900:].copy().reset_index(drop=True)

print(f"Evaluation dataset created with {len(sft_dataset_eval)} samples.")
print("Columns:", sft_dataset_eval.columns.tolist())
print("\nFirst 5 rows of the evaluation set:")
sft_dataset_eval.head()

## 4. Utility Functions
These are the core functions for creating prompts, parsing model outputs, and calculating our metrics.

In [ ]:
def create_evaluation_prompt(job_skills_str, candidate_eval_json):
    """Creates the prompt for the model based on a row from the dataset."""
    eval_data = json.loads(candidate_eval_json)
    candidate_skills = eval_data.get('skills_match', {}).get('present_skills', [])
    candidate_skills_str = ", ".join(candidate_skills)

    prompt = f"""<|im_start|>system
You are an HR expert evaluating candidate resumes. Provide a score (0-100), explanation, and status (SELECTED/REJECTED).
<|im_end|>
<|im_start|>user
Job Requirements: {job_skills_str}

Candidate Skills: {candidate_skills_str}

Evaluate this candidate and provide your assessment in JSON format:
{{"score": [0-100], "status": "SELECTED" or "REJECTED", "explanation": "your reasoning"}}
<|im_end|>
<|im_start|>assistant"""
    return prompt

def parse_model_output(generated_text):
    """Robustly parses the JSON output from the model's generated text."""
    try:
        # The model should output JSON after the assistant tag.
        # Look for the first curly brace to start parsing.
        json_match = re.search(r'\{[\s\S]*\}', generated_text)
        if json_match:
            json_str = json_match.group(0)
            data = json.loads(json_str)
            return {
                'predicted_status': str(data.get('status', 'REJECTED')).upper(),
                'predicted_score': int(data.get('score', 0)),
                'predicted_explanation': str(data.get('explanation', ''))
            }
    except (json.JSONDecodeError, AttributeError, ValueError):
        # Fallback if JSON is malformed or not found
        pass
    
    # Default fallback if parsing fails completely
    return {
        'predicted_status': 'REJECTED',
        'predicted_score': 0,
        'predicted_explanation': ''
    }

def calculate_metrics(ground_truth_df, predictions_list):
    """Calculates all performance metrics based on predictions."""
    # --- Prepare Ground Truth --- 
    # Status: Normalize status labels (e.g., 'STRONGLY_CONSIDER' -> 'SELECTED')
    status_mapping = {'CONSIDER': 'SELECTED', 'STRONGLY_CONSIDER': 'SELECTED'}
    true_status = ground_truth_df['status'].replace(status_mapping).tolist()
    # Score & Explanation: Parse from the ground truth JSON
    true_scores = [json.loads(x).get('overall_score', 0) for x in ground_truth_df['evaluation_json']]
    true_explanations = [json.loads(x).get('skills_match', {}).get('explanation', '') for x in ground_truth_df['evaluation_json']]

    # --- Prepare Predictions ---
    pred_status = [p['predicted_status'] for p in predictions_list]
    pred_scores = [p['predicted_score'] for p in predictions_list]
    pred_explanations = [p['predicted_explanation'] for p in predictions_list]

    # 1. Classification Metrics
    precision, recall, f1, _ = precision_recall_fscore_support(true_status, pred_status, average='binary', pos_label='SELECTED', zero_division=0)
    accuracy = accuracy_score(true_status, pred_status)

    # 2. Regression Metrics
    mae = np.mean(np.abs(np.array(true_scores) - np.array(pred_scores)))
    rmse = np.sqrt(np.mean((np.array(true_scores) - np.array(pred_scores))**2))

    # 3. Text Generation Metrics
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rouge_scores = [scorer.score(gt, pred)['rougeL'].fmeasure for gt, pred in zip(true_explanations, pred_explanations)]
    avg_rouge_l = np.mean(rouge_scores)
    
    return {
        "F1-Score": f1,
        "MAE (Score)": mae,
        "ROUGE-L (Explanation)": avg_rouge_l,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "RMSE (Score)": rmse
    }

## 5. Model Loading
We'll load the base Qwen2 model and then apply the SFT and GRPO LoRA adapters to create our two evaluation models.

In [ ]:
def load_model_with_adapter(base_model_name, adapter_path):
    """Loads the base model and applies a specific LoRA adapter."""
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=base_model_name,
        max_seq_length=2048,
        dtype=None,
        load_in_4bit=True,
    )
    tokenizer.pad_token = tokenizer.eos_token
    
    print(f"Loading adapter from: {adapter_path}")
    model = PeftModel.from_pretrained(model, adapter_path)
    print("✅ Model and adapter loaded successfully.")
    return model, tokenizer

# Load the SFT-only model
sft_model, tokenizer = load_model_with_adapter(BASE_MODEL_NAME, SFT_ADAPTER_PATH)

# Load the final GRPO model
grpo_model, _ = load_model_with_adapter(BASE_MODEL_NAME, GRPO_ADAPTER_PATH) # We can reuse the tokenizer

## 6. Generate Predictions
Now, we'll loop through our test set and generate predictions for both models. This may take a few minutes.

In [ ]:
def generate_predictions(model, tokenizer, test_df):
    """Generates predictions for an entire dataframe."""
    predictions = []
    # Using tqdm for a progress bar
    for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
        prompt = create_evaluation_prompt(row['job_description_skills'], row['evaluation_json'])
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.eos_token_id
            )
        
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        # We only care about the response after the final assistant tag
        assistant_response = generated_text.split('<|im_start|>assistant')[-1].strip()
        
        parsed_output = parse_model_output(assistant_response)
        predictions.append(parsed_output)
        
    return predictions

# Generate for SFT model
print("Generating predictions for SFT-Only Model...")
sft_predictions = generate_predictions(sft_model, tokenizer, sft_dataset_eval)

# Generate for GRPO model
print("\nGenerating predictions for Final GRPO Model...")
grpo_predictions = generate_predictions(grpo_model, tokenizer, sft_dataset_eval)

print("\n✅ All predictions generated!")

## 7. Calculate Metrics and Visualize Results
Finally, we'll use our `calculate_metrics` function and display the results in a clean comparison table.

In [ ]:
# Calculate metrics for both models
sft_results = calculate_metrics(sft_dataset_eval, sft_predictions)
grpo_results = calculate_metrics(sft_dataset_eval, grpo_predictions)

# Combine results into a pandas DataFrame for easy visualization
results_df = pd.DataFrame({
    'SFT-Only Model': sft_results,
    'GRPO Model (Ours)': grpo_results
})

print("--- Model Performance Comparison ---")
display(results_df.round(4))

### Qualitative Comparison: Example Output
Let's look at a single example to see the difference in output quality.

In [ ]:
sample_index = 42 # You can change this index to explore different samples

print(f"--- Comparing Outputs for Sample Index: {sample_index} ---")

# Ground Truth
ground_truth_row = sft_dataset_eval.iloc[sample_index]
true_eval_data = json.loads(ground_truth_row['evaluation_json'])
print("\n--- Ground Truth ---")
print(f"Job Skills: {ground_truth_row['job_description_skills']}")
print(f"True Status: {ground_truth_row['status']}")
print(f"True Score: {true_eval_data.get('overall_score')}")
print(f"True Explanation: {true_eval_data.get('skills_match', {}).get('explanation')}")

# SFT Model Prediction
sft_pred = sft_predictions[sample_index]
print("\n--- SFT-Only Model Prediction ---")
print(f"Predicted Status: {sft_pred['predicted_status']}")
print(f"Predicted Score: {sft_pred['predicted_score']}")
print(f"Predicted Explanation: {sft_pred['predicted_explanation']}")

# GRPO Model Prediction
grpo_pred = grpo_predictions[sample_index]
print("\n--- GRPO Model Prediction ---")
print(f"Predicted Status: {grpo_pred['predicted_status']}")
print(f"Predicted Score: {grpo_pred['predicted_score']}")
print(f"Predicted Explanation: {grpo_pred['predicted_explanation']}")

In [4]:
import csv, json

csv_file = "synthetic_resume_evaluation.csv"
json_file = "synthetic_resume_evaluation.json"

with open(csv_file, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    data = []
    for row in reader:
        # evaluation_json is already a JSON string—keep as string
        # Convert numeric fields back to correct types
        row["similarity_score"] = float(row["similarity_score"])
        row["confidence_score"] = float(row["confidence_score"])
        row["processing_duration_ms"] = int(row["processing_duration_ms"])
        row["time"] = float(row["time"])
        data.append(row)

with open(json_file, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2)


In [6]:
import json
import random

input_json = "synthetic_resume_evaluation.json"
output_json = "eval_data.json"

# Load all records
with open(input_json, "r", encoding="utf-8") as f:
    data = json.load(f)

# Randomly select 100 distinct records
sampled = random.sample(data, 100)

# Save the subset
with open(output_json, "w", encoding="utf-8") as f:
    json.dump(sampled, f, indent=2)

print(f"Random 100 records saved to {output_json}")


Random 100 records saved to eval_data.json
